# Lab07.3 — Real-Image Application and ARM–FPGA Benchmark

**Objective:** compare a scalar Python loop, vectorized NumPy and FPGA thresholding with AXI DMA. Then build a **hybrid ARM–FPGA color-selection pipeline** for the Logitech C270.

**Processing split:** the FPGA implements **only** a threshold on one 8-bit channel. For the camera example, this channel is **HSV saturation (S)**. ARM runs camera acquisition, BGR→HSV conversion, **hue (H) selection**, combination of masks, morphology and contour detection. A saturation mask by itself is **not a color detector**: it accepts many different hues.

**Practical demonstration:** show a strongly colored **blue object** (e.g. a blue card) in front of the camera for Step 8. You can later change `TARGET_COLOR` to `'red'` or `'green'`. The mask should reject unrelated regions; without an object of the selected color, **zero detections is a correct outcome**.

The notebook does not assume that FPGA is faster than NumPy; interpret the measurements produced on your own board.

### Step 1 — Load the FPGA design and sample image

Import the common driver from 07.2, reload `lab07.bit` and inspect DMA capabilities. Confirm that this is a hardware experiment and not a software simulation.

In [ ]:
from pathlib import Path
import sys, time, statistics
import numpy as np
import cv2
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path('../tools').resolve()))
from dma_threshold import ThresholdDMA, reference_threshold

source = cv2.imread('../images/scene_256.png', cv2.IMREAD_GRAYSCALE)
if source is None: raise FileNotFoundError('Missing Lab07/images/scene_256.png')
engine = ThresholdDMA()
print('DMA maximum bytes:', engine.dma.buffer_max_size)
print('Input image:', source.shape)

### Step 2 — Show CPU alternatives and validate identical arithmetic

NumPy operates on whole arrays in compiled/vectorized code. A simple Python loop is also included to illustrate interpreter overhead, **not** as a realistic optimized ARM baseline. All implementations use **pixel ≥ 128 ⇒ 255**.

In [ ]:
def scalar_python_threshold(gray, threshold=128):
    result = np.empty_like(gray)
    for y in range(gray.shape[0]):
        for x in range(gray.shape[1]):
            result[y, x] = 255 if int(gray[y,x]) >= threshold else 0
    return result

small = cv2.resize(source, (64,64), interpolation=cv2.INTER_AREA)
scalar_mask = scalar_python_threshold(small, 128)
numpy_mask = reference_threshold(small, 128)
fpga_mask, _ = engine.process(small, 128)
assert np.array_equal(scalar_mask, numpy_mask)
assert np.array_equal(fpga_mask, numpy_mask)
print('PASS — all three implementations agree on a 64 x 64 image')

### Step 3 — Benchmark image sizes and repeat the measurements

For each 64×64, 128×128, 256×256 and 512×512 image, measure **median** elapsed time for NumPy and FPGA+DMA, including buffer allocation and copying. Run the scalar Python loop at 64×64 and 128×128 only to avoid unnecessarily long waits. Repeat the trials to reduce transient Linux scheduling effects.

In [ ]:
sizes = [64, 128, 256, 512]
repeats = 5
rows = []
for size in sizes:
    picture = cv2.resize(source, (size,size), interpolation=cv2.INTER_LINEAR)
    expected = reference_threshold(picture, 128)
    sw_us, fpga_us, dma_only_us = [], [], []
    py_us = []
    for _ in range(repeats):
        tic = time.perf_counter_ns()
        mask_np = reference_threshold(picture, 128)
        sw_us.append((time.perf_counter_ns() - tic)/1000.)
        actual, timing = engine.process(picture, 128)
        fpga_us.append(timing['total_us'])
        dma_only_us.append(timing['dma_us'])
        assert np.array_equal(mask_np, expected) and np.array_equal(actual, expected)
        if size <= 128:
            tic = time.perf_counter_ns()
            mask_python = scalar_python_threshold(picture, 128)
            py_us.append((time.perf_counter_ns() - tic)/1000.)
            assert np.array_equal(mask_python, expected)
    rows.append((size, statistics.median(sw_us), statistics.median(fpga_us),
                 statistics.median(dma_only_us), statistics.median(py_us) if py_us else None))

print('{:>8} {:>15} {:>18} {:>16} {:>17}'.format('Side', 'NumPy (us)', 'FPGA total (us)', 'DMA only (us)', 'Python loop (us)'))
for side, nptime, total, dma, py in rows:
    py_string = '{:.1f}'.format(py) if py is not None else '(not tested)'
    print('{:8d} {:15.1f} {:18.1f} {:16.1f} {:>17}'.format(side, nptime, total, dma, py_string))

### Step 4 — Plot performance without presupposing a winner

The FPGA curve includes host-visible buffer handling and DMA, which is the relevant end-to-end service time for a real image. The NumPy baseline is compiled/vectorized, so it may be faster than a dedicated FPGA threshold; **interpret your measured values rather than asserting hardware speedup**.

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
pix = [r[0]**2 for r in rows]
ax.plot(pix, [r[1] for r in rows], 'o-', label='ARM NumPy (end-to-end mask)')
ax.plot(pix, [r[2] for r in rows], 's-', label='FPGA + DMA + host copies')
ax.plot(pix, [r[3] for r in rows], '^-', label='DMA portion observed by host')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Pixels per image'); ax.set_ylabel('Time (microseconds)')
ax.set_title('Threshold performance: measured, not theoretical')
ax.grid(True, which='both', alpha=0.25); ax.legend()
plt.show()

### Step 5 — Compare the ratios FPGA/NumPy and full-system throughput

A ratio greater than one means **the FPGA call is slower than NumPy**, not that its internal comparator is slower. Throughput includes the complete end-to-end call and should not be compared to the ideal four-pixels-per-clock capacity without acknowledging memory and software overhead.

In [ ]:
print('{:>8} {:>16} {:>20}'.format('Pixels', 'FPGA/NumPy time', 'FPGA MiB/s'))
for side, numpy_us, total_us, dma_us, _ in rows:
    ratio = total_us/numpy_us
    mib_s = (side*side/2**20)/(total_us/1e6)
    print('{:8d} {:16.2f} {:20.2f}'.format(side*side, ratio, mib_s))

## Camera application — Logitech C270 (optional when camera unavailable)

### Step 6 — Acquire **one** camera frame safely

The C270 uses `/dev/video0` in the typical setup. Open it with V4L2, request 320×240, read a frame and always release the device in a `finally` block. No continuous capture loop is required to understand this experiment.

In [ ]:
camera_frame = None
cap = cv2.VideoCapture(0, cv2.CAP_V4L2)
try:
    if not cap.isOpened():
        print('Camera is unavailable; skip the camera cells. Synthetic-image benchmark is complete.')
    else:
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 320)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 240)
        for _ in range(4):
            ok, frame = cap.read()
            if ok and frame is not None:
                camera_frame = frame
        if camera_frame is None:
            print('Camera opened but did not return a usable frame.')
        else:
            print('Captured frame:', camera_frame.shape)
finally:
    cap.release()

if camera_frame is not None:
    plt.figure(figsize=(7,4))
    plt.imshow(cv2.cvtColor(camera_frame, cv2.COLOR_BGR2RGB))
    plt.axis('off'); plt.title('Logitech C270 frame'); plt.show()

### Step 7 — Compute the saturation threshold in FPGA

1. ARM captures the image, resizes it to 320×240 and converts BGR→HSV.
2. The **S channel** represents color saturation, not brightness and not color identity. V represents brightness; H represents hue (red, green, blue, etc.).
3. The FPGA generates `S >= 110` as a binary mask (`255` for accepted pixels, otherwise `0`).
4. We check the FPGA mask against NumPy **pixel by pixel**.

**Important:** a high-saturation mask may include clothes, furniture, lighting artifacts or several differently colored objects. It cannot determine which pixels are *blue* or *red*. We use the H channel in Step 8 to select a specific color. The combination is a straightforward introduction to **PS–PL co-processing**.

In [ ]:
camera_saturation = None
camera_mask = None
if camera_frame is not None:
    camera_small = cv2.resize(camera_frame, (320,240), interpolation=cv2.INTER_AREA)
    hsv = cv2.cvtColor(camera_small, cv2.COLOR_BGR2HSV)
    camera_saturation = hsv[:, :, 1].copy()
    camera_mask, timing = engine.process(camera_saturation, 110)
    expected = reference_threshold(camera_saturation, 110)
    assert np.array_equal(camera_mask, expected)
    print('Camera saturation mask matches NumPy:', camera_mask.shape)
    print('FPGA threshold total time: {:.1f} us'.format(timing['total_us']))
    fig, axes = plt.subplots(1,3,figsize=(12,4))
    axes[0].imshow(cv2.cvtColor(camera_small, cv2.COLOR_BGR2RGB)); axes[0].set_title('Camera image')
    axes[1].imshow(camera_saturation,cmap='gray',vmin=0,vmax=255); axes[1].set_title('HSV saturation — ARM')
    axes[2].imshow(camera_mask,cmap='gray',vmin=0,vmax=255); axes[2].set_title('Threshold mask — FPGA')
    for ax in axes:ax.axis('off')
    plt.show()
else:
    print('Camera not available: no hardware camera frame to process.')

### Step 8 — Select one COLOR and locate its object (hybrid ARM–FPGA)

A threshold on **HSV saturation S alone** cannot distinguish blue from red or green: it only separates strongly saturated pixels from weakly saturated pixels. This is why the earlier demo selected unrelated patches on the scene.

In this corrected experiment:

1. **FPGA:** calculates the saturation mask (`S >= 110`) in Step 7, using AXI DMA.
2. **ARM:** creates a *hue* mask for a chosen target (`blue` by default). OpenCV hue values range from 0 to 179, rather than 0 to 360 degrees; red wraps around the end of the interval.
3. **ARM:** combines the FPGA mask with the hue mask and excludes very dark pixels (`V >= 50`).
4. **ARM:** removes isolated mask noise, extracts contours and keeps components larger than 250 pixels.
5. We display the **original frame**, **FPGA saturation mask**, **ARM hue mask**, **combined color mask**, **cleaned color mask** and **detected regions** side by side.

**To test:** hold a clearly visible **blue piece of paper or other blue object** in front of the Logitech C270. The detection rectangle should mark that object rather than arbitrary regions of the background. If no blue object is visible, *0 detections* is expected, not an error. For a different object, change `TARGET_COLOR` in the following cell.

In [ ]:
# Select the color to be detected, independently of the FPGA saturation threshold.
TARGET_COLOR = 'blue'            # Choose: 'blue', 'green' or 'red'
MIN_AREA_PX = 250               # Ignore small regions/noise
MIN_VALUE = 50                  # Reject near-black pixels on ARM

# OpenCV represents H in [0,179]. Red requires two ranges around the wrap-around.
HUE_RANGES = {
    'blue':  [(95, 130)],
    'green': [(35, 85)],
    'red':   [(0, 10), (170, 179)],
}

if TARGET_COLOR not in HUE_RANGES:
    raise ValueError('TARGET_COLOR must be blue, green or red')

if camera_mask is not None:
    # camera_small and hsv come from Step 7; camera_mask came from FPGA via DMA.
    hue = hsv[:, :, 0]
    value = hsv[:, :, 2]
    hue_mask = np.zeros(hue.shape, dtype=np.uint8)
    for low, high in HUE_RANGES[TARGET_COLOR]:
        hue_mask = cv2.bitwise_or(hue_mask, cv2.inRange(hue, low, high))

    value_mask = cv2.inRange(value, MIN_VALUE, 255)
    combined = cv2.bitwise_and(camera_mask, hue_mask)  # S threshold: FPGA; H: ARM
    combined = cv2.bitwise_and(combined, value_mask)   # Bright enough: ARM

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    cleaned = cv2.morphologyEx(combined, cv2.MORPH_OPEN, kernel)
    contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL,
                                    cv2.CHAIN_APPROX_SIMPLE)

    result = camera_small.copy()
    boxes = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if area >= MIN_AREA_PX:
            x, y, w, h = cv2.boundingRect(contour)
            boxes.append((x, y, w, h, round(area)))
            cv2.rectangle(result, (x, y), (x + w, y + h), (0, 255, 255), 2)

    print('Selected hue:', TARGET_COLOR)
    print('FPGA threshold: saturation >= 110')
    print('ARM thresholds: hue in', HUE_RANGES[TARGET_COLOR],
          ', value >=', MIN_VALUE)
    print('Objects matching selected color:', len(boxes))
    print('Bounding boxes (x, y, w, h, area):', boxes)

    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    panels = [
        (cv2.cvtColor(camera_small, cv2.COLOR_BGR2RGB), 'Camera input (ARM)', None),
        (camera_mask, 'FPGA mask: saturation S', 'gray'),
        (hue_mask, 'ARM mask: target hue H', 'gray'),
        (combined, 'Combined mask: S AND H AND V', 'gray'),
        (cleaned, 'Morphology and contours: ARM', 'gray'),
        (cv2.cvtColor(result, cv2.COLOR_BGR2RGB),
         'Selected-color bounding boxes: ARM', None),
    ]
    for ax, (img, title, cmap) in zip(axes.flat, panels):
        if cmap:
            ax.imshow(img, cmap=cmap, vmin=0, vmax=255)
        else:
            ax.imshow(img)
        ax.set_title(title, fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No camera frame; color selection demonstration skipped.')

## Student questions and conclusions

1. Is the FPGA+DMA path faster than vectorized NumPy for **your measured values**? What costs are included in the reported FPGA time?
2. What is the purpose of the three HSV channels **H**, **S** and **V**?
3. Why does the FPGA **saturation** mask alone detect many unrelated colored regions?
4. Which steps in the corrected color pipeline run on **FPGA** and which on **ARM**?
5. What happens when the blue object is removed from the frame? Why is zero detections then a valid result?
6. Why do morphology and the minimum-area threshold help suppress isolated noise without guaranteeing object recognition?
7. How might we increase performance: reusing DMA buffers, processing several operations in one FPGA pass, or adding hue selection directly to the hardware?

**Conclusion:** FPGA thresholding is a simple data-parallel operation. A useful application requires additional information (here, the hue channel) and several ARM-side processing steps. The exercise demonstrates correct **ARM–FPGA partitioning**, not general-purpose object recognition.